<style>
div.mermaid > svg { width: 70% !important; height: auto !important; }
</style>

# Prefix sum (warp-shuffle scan)

Our first **non-deep-learning** primitive. A *scan* turns a row `[a, b, c, d]` into its running
totals `[a, a+b, a+b+c, a+b+c+d]`. It is the quiet engine behind sorting, stream compaction, and
histograms.

Each output depends on every element before it, so a scan looks stubbornly *sequential*. The key
insight of this notebook is that it isn't: the very same **two-level (warp → shared memory)**
reduction you built for softmax also parallelizes a scan. The only twist is that we keep *every*
partial sum along the way instead of collapsing the row down to a single number.

**You'll learn:** the inclusive parallel scan; the **Hillis-Steele warp scan** built from
`cute.arch.shuffle_sync_up` in log₂(32) = 5 steps -- factored into a small reusable `@cute.jit`
helper; and how to stitch per-warp scans into a block-wide scan with one shared-memory pass.

**Runs on:** any CUDA GPU. **Prereq:** the `01_softmax` notebook (same two-level reduction).

In [ ]:
import cutlass
import cutlass.cute as cute
import torch

## 1. A scan in two levels

One block per row, one thread per element, so `BLOCK_SIZE == C`. The strategy mirrors the softmax
reduction, but a scan must keep *every* intermediate sum instead of collapsing the row to one
number.

```text
 1. WARP SCAN    each warp inclusive-scans its 32 lanes with shuffle_sync_up (offsets 1,2,4,8,16)
 2. WARP TOTALS  the last lane of each warp writes its running total to shared memory
 3. SCAN TOTALS  warp 0 scans those per-warp totals (the very same warp scan)
 4. ADD PREFIX   each warp adds the totals of all earlier warps -> the full row scan
```

The engine of step 1 is `shuffle_sync_up(v, d)`, which hands a lane the value held `d` lanes below
it. A lane folds that value in only when `lane >= d`; otherwise there is no real neighbor that far
down. Doubling the distance reaches every lane below in log₂(32) = 5 steps:

| step | offset | a lane now holds |
|---|---|---|
| 1 | 1 | sum of itself + 1 lane below |
| 2 | 2 | sum of the 4 nearest lanes |
| 3 | 4 | sum of the 8 nearest lanes |
| 4 | 8 | sum of the 16 nearest lanes |
| 5 | 16 | sum of all 32 lanes at and below it |

Steps 1 and 3 run the **identical** warp scan, so we write it once as a `@cute.jit` helper and call
it at both levels -- a reusable *device subroutine*, the kind of factoring you'll lean on in the
Blackwell chapters.

In [ ]:
@cute.jit
def warp_inclusive_scan(val, lane_id):
    """Inclusive prefix sum up the 32 lanes of a warp (Hillis-Steele, 5 shuffle steps)."""
    for offset in [1, 2, 4, 8, 16]:
        below = cute.arch.shuffle_sync_up(val, offset)  # value held `offset` lanes below
        if lane_id >= offset:                     # fold in only a real neighbor
            val = val + below
    return val

In [ ]:
@cute.kernel
def prefix_sum_kernel(
    inp: cutlass.Array, out: cutlass.Array, C: cutlass.Constexpr, BLOCK_SIZE: cutlass.Constexpr
):
    WARP_SIZE = 32
    WARPS_PER_BLOCK = BLOCK_SIZE // WARP_SIZE
    warp_totals = cutlass.Array(cutlass.Float32, WARPS_PER_BLOCK, space=cutlass.AddressSpace.smem)

    row, _, _ = cute.arch.block_idx()  # one block per row
    tid, _, _ = cute.arch.thread_idx()
    warp_id = tid // WARP_SIZE
    lane_id = tid % WARP_SIZE
    base = row * C                     # flat (N, C): one element per thread (C == BLOCK_SIZE)

    # Step 1. Inclusive scan within the warp (the helper).
    running_sum = warp_inclusive_scan(inp[base + tid], lane_id)

    # Step 2. The last lane holds the warp total; publish it for the cross-warp scan.
    if lane_id == WARP_SIZE - 1:
        warp_totals[warp_id] = running_sum
    cute.arch.barrier()

    # Step 3. Warp 0 scans the per-warp totals with the SAME helper, so warp_totals[w]
    # becomes the sum of warps 0..w. Idle lanes carry 0.
    if warp_id == 0:
        warp_total = 0.0
        if lane_id < WARPS_PER_BLOCK:
            warp_total = warp_totals[lane_id]
        warp_total = warp_inclusive_scan(warp_total, lane_id)
        if lane_id < WARPS_PER_BLOCK:
            warp_totals[lane_id] = warp_total
    cute.arch.barrier()

    # Step 4. Lift each warp-local scan to the row scan by adding earlier warps' totals.
    if warp_id > 0:
        running_sum = running_sum + warp_totals[warp_id - 1]
    out[base + tid] = running_sum

## 2. Launch: one block per row

Grid `(N, 1, 1)` is one block per row; block `(C, 1, 1)` is one thread per column, so
`BLOCK_SIZE == C`. Because `C` is a `Constexpr`, the shuffle offsets and the warp count are known at
compile time, so the scan loops unroll into straight-line shuffle code.

In [ ]:
@cute.jit
def prefix_sum(
    inp: cutlass.Array,
    out: cutlass.Array,
    N: cutlass.Int32,
    C: cutlass.Constexpr,
):
    # One block per row, one thread per column (BLOCK_SIZE == C).
    prefix_sum_kernel(inp, out, C, C).launch(grid=(N, 1, 1), block=(C, 1, 1))

## 3. Run and check against PyTorch

`cutlass.Array` parameters accept the PyTorch CUDA tensors **directly** through
`cute.runtime.from_dlpack` -- no manual copy. The reference is `torch.cumsum`, which is exactly an
inclusive scan with addition.

In [ ]:
# Inputs live on the GPU and are handed to the kernel straight through DLPack.
N, C = 1024, 256
inp = torch.randn(N, C, dtype=torch.float32, device="cuda")
out = torch.zeros_like(inp)

prefix_sum(cute.runtime.from_dlpack(inp), cute.runtime.from_dlpack(out), N, C=C)

# torch.cumsum is the inclusive prefix sum; compare on the host.
torch.testing.assert_close(
    out.cpu(), torch.cumsum(inp.cpu(), dim=-1), atol=1e-2, rtol=1e-3
)
print("PASS")

## Try it yourself

1. **Exclusive scan** (`out[i] = sum(inp[0..i-1])`, with `out[0] = 0`): fold the neighbor in only
   when `lane > offset`, or shift the inclusive result right by one lane. Check it against a shifted
   `torch.cumsum`.
2. **A different operator:** swap `+` for `max` and the scan becomes a running maximum — the same
   structure with a different monoid. What identity value replaces the `0.0` seed in warp 0?
3. **Rows wider than a block (`C > BLOCK_SIZE`):** scan the row in tiles of `BLOCK_SIZE`, carrying
   each tile's grand total into the next. Which part is forced to be sequential, and which stays
   parallel?